In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.helpers import load_netflix_data, prepare_country_credits, prepare_genre_credits

df = load_netflix_data()

# column level profile
profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(dropna=True)
})
display(profile)

print(f"Dataset shape: {df.shape}")
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate show IDs: {df['show_id'].duplicated().sum()}")

,dtype,missing_values,unique_values
show_id,str,0,8790
type,str,0,2
title,str,0,8786
director,str,0,4528
country,str,0,86
date_added,str,0,1713
release_year,int64,0,74
rating,str,0,14
duration,str,0,220
listed_in,str,0,513


Dataset shape: (8790, 10)
Exact duplicate rows: 0
Duplicate show IDs: 0


In [2]:
unknown_country_labels = {"unknown country", "unknown", "not given", ""}
unknown_country_count = (
    df["country"]
    .astype("string")
    .str.strip()
    .str.lower()
    .isin(unknown_country_labels)
    .sum()
)
print(f"Titles with unknown country: {unknown_country_count}")

# check if dates can be parsed
added_dates = pd.to_datetime(df["date_added"], errors="coerce")
print(f"Unparsable date_added values: {added_dates.isna().sum()}")
print(f"date_added range: {added_dates.min()} to {added_dates.max()}")

display(df.head())

Titles with unknown country: 287
Unparsable date_added values: 0
date_added range: 2008-01-01 00:00:00 to 2021-09-25 00:00:00


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [3]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

type_counts = df["type"].value_counts()
type_percentages = (type_counts / len(df) * 100).round(1)
release_counts = df["release_year"].value_counts().sort_index()
added_by_year = df["date_added"].dt.year.value_counts().sort_index()

print(f"Movies: {type_counts.get('Movie', 0):,} ({type_percentages.get('Movie', 0)}%)")
print(f"TV Shows: {type_counts.get('TV Show', 0):,} ({type_percentages.get('TV Show', 0)}%)")
print(
    f"Most common release year: {release_counts.idxmax()}"
    f"({release_counts.max():,} titles)"
)
print(f"Most common rating: {df['rating'].value_counts().idxmax()}")
print(
    f"Most titles added in one calendar year: {added_by_year.idxmax()}"
    f"({added_by_year.max():,} titles)"
)

display(
    pd.DataFrame({
        "titles_added": added_by_year
    }).tail(10)
)

Movies: 6,126 (69.7%)
TV Shows: 2,664 (30.3%)
Most common release year: 2018(1,146 titles)
Most common rating: TV-MA
Most titles added in one calendar year: 2019(2,016 titles)


,titles_added
date_added,
2012,3
2013,11
2014,24
2015,82
2016,426
2017,1185
2018,1648
2019,2016
2020,1879


In [4]:
content_mix = (
    type_counts
    .rename_axis("content_type")
    .reset_index(name="title_count")
)

fig_mix = px.pie(
    content_mix,
    names = "content_type",
    values = "title_count",
    hole = 0.45,
    title = "Netflix Catalog Mix by Content Type"
)
fig_mix.update_traces(
    textposition = "inside",
    textinfo = "label + percent",
    hovertemplate = "%{label}<br>Titles: %{value:,}<br>Share: %{percent}<extra></extra>"
)
fig_mix.show()

In [5]:
added_data = (
    added_by_year
    .rename_axis("year")
    .reset_index(name="titles_added")
)

fig_added = px.line(
    added_data,
    x="year",
    y="titles_added",
    markers=True,
    title=(
        "Titles Added to the Catalog by Year"
        "<br><sup>2021 is partial: data ends September 25, 2021</sup>"
    ),
    labels={
        "year": "Year added",
        "titles_added": "Number of titles",
    },
)
fig_added.update_traces(
    hovertemplate="%{x}<br>Titles added: %{y:,}<extra></extra>"
)
fig_added.show()

In [6]:
recent_release_counts = release_counts.loc[2000:]

fig_release_trend = px.line(
    recent_release_counts.rename_axis("release_year").reset_index(name="title_count"),
    x="release_year",
    y="title_count",
    markers=True,
    title=(
        "Titles by Release Year (2000–2021)"
        "<br><sup>Counts reflect this dataset; the latest release years may be incomplete</sup>"
    ),
    labels={
        "release_year": "Release year",
        "title_count": "Titles in dataset",
    },
)
fig_release_trend.update_traces(
    hovertemplate="%{x}<br>Titles: %{y:,}<extra></extra>"
)
fig_release_trend.show()

In [7]:
# country credits 
country_titles, _ = prepare_country_credits(df)
country_counts = (
    country_titles.groupby("country")["show_id"]
    .nunique()
    .sort_values(ascending=False)
)

top_countries = (
    country_counts.head(15)
    .sort_values()
    .rename_axis("country")
    .reset_index(name="title_count")
)

fig_countries = px.bar(
    top_countries,
    x = "title_count",
    y = "country",
    orientation  = "h",
    title = "Top 15 Countries by Titles in the Dataset",
    labels = {"title_count": "Titles with country credit", "country": "Country"},
    text = "title_count",
)
fig_countries.update_layout(yaxis={"categoryorder": "total ascending"})
fig_countries.show()

In [8]:
# genre credits
genre_titles = prepare_genre_credits(df)

genre_counts = (
    genre_titles.groupby("genre")["show_id"]
    .nunique()
    .sort_values(ascending=False)
)

top_genres = (
    genre_counts.head(15)
    .sort_values()
    .rename_axis("genre")
    .reset_index(name="title_count")
)

fig_genres = px.bar(
    top_genres,
    x = "title_count",
    y = "genre",
    orientation = "h",
    title = "Top 15 Genre Labels",
    labels = {"title_count": "Titles with genre label", "genre": "Genre"},
    text = "title_count",
)
fig_genres.update_layout(yaxis={"categoryorder": "total ascending"})
fig_genres.show()

In [9]:
# rating composition
rating_counts = pd.crosstab(df["type"], df["rating"])
rating_percentages = (
    rating_counts.div(rating_counts.sum(axis=1), axis=0).mul(100).round(1)
)

rating_plot_data = (
    rating_percentages
    .rename_axis("content_type")
    .reset_index()
    .melt(
        id_vars="content_type",
        var_name="rating",
        value_name="percentage",
    )
)

fig_ratings = px.bar(
    rating_plot_data,
    x = "content_type",
    y = "percentage",
    color = "rating",
    barmode = "stack",
    title = "Rating Composition by Content Type",
    labels = {
        "content_type": "Content type",
        "percentage": "Share within content type (%)",
        "rating": "Rating",
    },
)
fig_ratings.show()

# Executive summary

This analysis covers 8,790 titles in the cleaned Netflix catalog snapshot. Movies account for 6,126 titles (69.7%), and TV Shows account for 2,664 (30.3%). The dataset describes catalog composition; it does not include viewership, revenue, or regional availability.

## Key findings

- **Content mix:** Movies make up about seven in ten titles in this snapshot.
- **Catalog additions:** 2019 has the most titles by `date_added` (2,016). The data ends on September 25, 2021, so 2021 is a partial year and should not be compared directly with full years.
- **Release years:** The largest release-year count is 2018 (1,146 titles). Counts fall after 2018, but this reflects release years in the current dataset—not a verified change in Netflix’s own production.
- **Geographic credits:** The United States leads (3,240 titles), followed by India (1,057) and the United Kingdom (638). Country credits are not the same as availability in those markets; 287 titles have an unknown-country placeholder. Multi-country titles contribute to each credited country.
- **Ratings and genres:** TV-MA is the most common rating overall (3,205 titles). The most frequent genre labels are International Movies (2,752), Dramas (2,426), and Comedies (1,674). Titles can have multiple genre labels, so genre counts overlap.

## Potential business actions

1. **Evaluate the movie-heavy mix:** Compare the catalog mix with viewing hours, completion rates, and retention by content type before changing acquisition or commissioning plans.
2. **Investigate international-market opportunities:** Review title performance, licensing costs, and availability by region—especially for titles with Indian and UK country credits—before prioritizing investment.
3. **Use genre and rating patterns to form testable hypotheses:** Explore audience engagement for drama, comedy, and TV-MA titles, and separately assess family-oriented programming. The current catalog counts alone cannot show demand.
4. **Improve completeness:** Track or resolve the 287 unknown-country records to make geographic reporting more reliable.

## Limitations

This is a static catalog snapshot, not a live inventory. `date_added` ends on September 25, 2021. The dataset contains no viewing, customer, revenue, licensing-cost, or market-availability measures; therefore, these findings describe catalog composition and suggest questions for follow-up analysis, rather than proving audience demand or business impact. Country and genre counts overlap when a title has multiple credits.